# Feature Engineering

1. Physics based features
2. Rolling/ Windows features
3. Interaction features
4. Machine type features
5. Threashold flags 
6. External context features

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

#& to handle input file(s)
from pathlib import Path

In [2]:

#? csv file path (to resolve problem at production level)
BASE_DIR = Path().resolve().parent
df = pd.read_csv(BASE_DIR / 'Dataset' / 'ai4i2020_cleaned.csv')
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,0,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,0,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:

#^ copy of baseline features
df_feat = df.copy()

print("\nOriginal features:", df_feat.columns.tolist())
print("\nShape:", df_feat.shape)


Original features: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Shape: (10000, 12)


# 1. Physical Features


In [4]:

#* power 
df_feat['power'] = df_feat['Torque [Nm]'] * df_feat['Rotational speed [rpm]']

#~ Heat stress (temp. difference)
df_feat['temp_diff'] = df_feat['Process temperature [K]'] - df_feat['Air temperature [K]']

#todo Wear rate (tool degradation per rotation)
df_feat['wear_rate'] = df_feat['Tool wear [min]'] / (df_feat['Rotational speed [rpm]'] + 1)

#? Torque / wear (stress on tool)
df_feat['torque_per_wear'] = df_feat['Torque [Nm]'] / (df_feat['Tool wear [min]'] + 1)

#& mechnical strain
df_feat['strain_index'] = df_feat['Torque [Nm]'] * df_feat['Tool wear [min]']

#^ efficiency : power per temprature
df_feat['power_per_temp'] = df_feat['power'] / df_feat['Process temperature [K]']

print("Physics features created!")
df_feat[['power','temp_diff','wear_rate','torque_per_wear','strain_index','power_per_temp']].describe()

Physics features created!


,power,temp_diff,wear_rate,torque_per_wear,strain_index,power_per_temp
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,59967.147040,10.000630,0.070918,1.324723,4314.664550,193.445095
std,10193.093881,1.001094,0.042590,4.735015,2826.567692,32.903588
min,10966.800000,7.600000,0.000000,0.026168,0.000000,35.606494
25%,53105.400000,9.300000,0.034520,0.239975,1963.650000,171.250988
50%,59883.900000,9.800000,0.069949,0.371277,4012.950000,193.124322
75%,66873.750000,11.000000,0.105608,0.735153,6279.000000,215.557913
max,99980.400000,12.100000,0.185273,68.500000,16497.000000,324.717116


In [5]:
df_feat.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,power,temp_diff,wear_rate,torque_per_wear,strain_index,power_per_temp
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,66382.8,10.5,0.000000,42.800000,0.0,215.109527
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,65190.4,10.5,0.002129,11.575000,138.9,211.177195
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,74001.2,10.4,0.003336,8.233333,247.0,239.874230
3,0,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,56603.5,10.4,0.004881,4.937500,276.5,183.420285
4,0,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,56320.0,10.5,0.006388,4.000000,360.0,182.442501


# 2. Rolling/ Windowa features

analyze trends by using rolling mean and std. deviations

In [6]:

#! Rolling window size
W_SHORT = 5
W_LONG  = 10

#? RPM rolling features
df_feat['rpm_rolling_mean'] = df_feat['Rotational speed [rpm]'].rolling(W_SHORT, min_periods=1).mean()
df_feat['rpm_rolling_std']  = df_feat['Rotational speed [rpm]'].rolling(W_SHORT, min_periods=1).std().fillna(0) 

#^ Torque 
df_feat['torque_rolling_mean'] = df_feat['Torque [Nm]'].rolling(W_SHORT, min_periods=1).mean()
df_feat['torque_rolling_std']  = df_feat['Torque [Nm]'].rolling(W_SHORT, min_periods=1).std().fillna(0)

#todo Temperature rolling features
df_feat['temp_rolling_mean'] = df_feat['Air temperature [K]'].rolling(W_SHORT, min_periods=1).mean()
df_feat['temp_rolling_std']  = df_feat['Air temperature [K]'].rolling(W_SHORT, min_periods=1).std().fillna(0)

#* Tool wear rolling (longer window — wear accumulates slowly)
df_feat['wear_rolling_mean'] = df_feat['Tool wear [min]'].rolling(W_LONG, min_periods=1).mean()

#& Power rolling 
df_feat['power_rolling_mean'] = df_feat['power'].rolling(W_SHORT, min_periods=1).mean()
df_feat['power_rolling_std']  = df_feat['power'].rolling(W_SHORT, min_periods=1).std().fillna(0)

print("Rolling features created!")
df_feat[['rpm_rolling_mean','torque_rolling_mean','power_rolling_mean','wear_rolling_mean']].describe()

Rolling features created!


,rpm_rolling_mean,torque_rolling_mean,power_rolling_mean,wear_rolling_mean
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,1538.774907,39.987646,59968.299790,107.943520
std,81.852540,4.535626,4621.690518,56.306599
min,1322.000000,23.820000,43669.640000,0.000000
25%,1481.800000,36.960000,56887.830000,59.300000
50%,1527.600000,39.940000,59931.370000,107.700000
75%,1581.200000,43.040000,63034.975000,156.000000
max,2004.400000,56.920000,77000.240000,240.700000


# 3. OUTLIER ANALYSIS 
Finding outliers and handling them 

In [8]:
# IQR outlier flag function
def iqr_outlier_flag(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    return ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).astype(int)

# Apply only to RPM and Torque — only columns with meaningful outliers
df_feat['rpm_outlier_flag']    = iqr_outlier_flag(df_feat, 'Rotational speed [rpm]')
df_feat['torque_outlier_flag'] = iqr_outlier_flag(df_feat, 'Torque [Nm]')

# Validate — failure rate in outlier vs normal rows
for col, flag in [('Rotational speed [rpm]', 'rpm_outlier_flag'),
                  ('Torque [Nm]',            'torque_outlier_flag')]:
    out  = df_feat[df_feat[flag] == 1]['Machine failure'].mean() * 100
    norm = df_feat[df_feat[flag] == 0]['Machine failure'].mean() * 100
    print(f'{col}')
    print(f'  Outlier rows failure rate : {out:.2f}%')
    print(f'  Normal  rows failure rate : {norm:.2f}%\n')

Rotational speed [rpm]
  Outlier rows failure rate : 8.37%
  Normal  rows failure rate : 3.17%

Torque [Nm]
  Outlier rows failure rate : 89.86%
  Normal  rows failure rate : 2.79%



In [ ]:
# Z-Score Method (secondary — good for near-normal features like temp)
from scipy import stats

zscore_flags = pd.DataFrame(index=df_feat.index)

for col in SENSOR_FEATURES:
    z_scores = np.abs(stats.zscore(df_feat[col]))
    zscore_flags[f'{col}_zscore_outlier'] = (z_scores > 3).astype(int)
    print(f"{col}: outliers={zscore_flags[f'{col}_zscore_outlier'].sum()}")

Air temperature [K]: outliers=0
Process temperature [K]: outliers=0
Rotational speed [rpm]: outliers=164
Torque [Nm]: outliers=25
Tool wear [min]: outliers=0


In [10]:
print(df_feat['Machine failure'].value_counts(normalize=True))


Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64
